In [0]:
USE workspace.gold;

CREATE OR REPLACE TABLE dim_property AS
SELECT
    ROW_NUMBER() OVER (ORDER BY block, street_name) AS property_key,

    block,
    street_name,
    MAX(town) AS town,

    MAX(max_floor) AS max_floor,
    MAX(year_completed) AS year_completed,
    MAX(residential_flag) AS residential_flag,
    MAX(commercial_flag) AS commercial_flag,
    MAX(market_hawker_flag) AS market_hawker_flag,
    MAX(carpark_flag) AS carpark_flag,
    MAX(total_dwelling_units) AS total_dwelling_units

FROM workspace.silver.property_clean
GROUP BY block, street_name;

num_affected_rows,num_inserted_rows


In [0]:
USE workspace.gold;
CREATE OR REPLACE TABLE dim_date
USING DELTA AS
WITH unique_dates AS (
  SELECT DISTINCT transaction_date, year, month_num
  FROM workspace.silver.resale_clean
)
SELECT
  ROW_NUMBER() OVER (ORDER BY transaction_date) AS date_key,
  transaction_date AS full_date,
  year, month_num
FROM unique_dates;

num_affected_rows,num_inserted_rows


In [0]:
USE workspace.gold;
CREATE OR REPLACE TABLE dim_flat_type AS
WITH flat_types AS (
    SELECT DISTINCT trim(flat_type) AS flat_type
    FROM (
        SELECT flat_type FROM workspace.silver.resale_clean
        UNION
        SELECT flat_type FROM workspace.silver.rental_clean
        UNION
        SELECT flat_type FROM workspace.silver.property_flat_supply
    )
    WHERE flat_type IS NOT NULL
)
SELECT
    ROW_NUMBER() OVER (ORDER BY flat_type) AS flat_type_key,
    flat_type
FROM flat_types;

num_affected_rows,num_inserted_rows


In [0]:
USE workspace.gold;

CREATE OR REPLACE TABLE fact_resale
USING DELTA AS
SELECT
    p.property_key,
    d.date_key,
    f.flat_type_key,

    r.property_age,
    r.resale_price,
    r.floor_area_sqm,
    r.price_per_sqm

FROM workspace.silver.resale_clean r

LEFT JOIN dim_property p
    ON r.block = p.block
    AND r.street_name = p.street_name

LEFT JOIN dim_date d
    ON r.transaction_date = d.full_date

LEFT JOIN dim_flat_type f
    ON r.flat_type = f.flat_type;

num_affected_rows,num_inserted_rows


In [0]:
USE workspace.gold;SELECT COUNT(*) FROM fact_resale;


COUNT(*)
228542


In [0]:
SELECT COUNT(*) FROM workspace.silver.resale_clean;

COUNT(*)
228542


In [0]:
USE workspace.gold;
CREATE OR REPLACE TABLE fact_rental
USING DELTA AS
SELECT
    p.property_key,
    f.flat_type_key,
    d.date_key,

    r.monthly_rent

FROM workspace.silver.rental_clean r

LEFT JOIN dim_property p
    ON r.block = p.block
    AND r.street_name = p.street_name

LEFT JOIN dim_flat_type f
    ON r.flat_type = f.flat_type

LEFT JOIN dim_date d
    ON r.rent_approval_date = d.full_date;

num_affected_rows,num_inserted_rows


In [0]:
USE workspace.gold;
CREATE OR REPLACE TABLE fact_supply
USING DELTA AS
SELECT
    p.property_key,
    f.flat_type_key,

    SUM(s.units_sold) AS total_units_sold

FROM workspace.silver.property_flat_supply s

LEFT JOIN dim_property p
    ON s.block = p.block
    AND s.street_name = p.street_name

LEFT JOIN dim_flat_type f
    ON s.flat_type = f.flat_type

GROUP BY p.property_key, f.flat_type_key;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE investment_metrics
USING DELTA AS
SELECT
    r.town,

    AVG(r.resale_price) AS avg_price,
    AVG(r.price_per_sqm) AS avg_price_sqm,

    COUNT(*) AS demand,

    AVG(rt.monthly_rent) AS avg_rent,

    SUM(s.units_sold) AS total_supply

FROM workspace.silver.resale_clean r

LEFT JOIN workspace.silver.rental_clean rt
    ON r.block = rt.block
    AND r.street_name = rt.street_name
    AND r.flat_type = rt.flat_type

LEFT JOIN workspace.silver.property_flat_supply s
    ON r.block = s.block
    AND r.street_name = s.street_name

GROUP BY r.town;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE investment_score AS
SELECT
    town,

    avg_price_sqm,
    demand,
    avg_rent,
    total_supply,

    -- Normalize
    (avg_price_sqm - MIN(avg_price_sqm) OVER()) /
    (MAX(avg_price_sqm) OVER() - MIN(avg_price_sqm) OVER()) AS price_score,

    (demand - MIN(demand) OVER()) /
    (MAX(demand) OVER() - MIN(demand) OVER()) AS demand_score,

    (avg_rent - MIN(avg_rent) OVER()) /
    (MAX(avg_rent) OVER() - MIN(avg_rent) OVER()) AS rent_score,

    (total_supply - MIN(total_supply) OVER()) /
    (MAX(total_supply) OVER() - MIN(total_supply) OVER()) AS supply_score

FROM investment_metrics;

num_affected_rows,num_inserted_rows


In [0]:
SELECT *,
    (0.4 * price_score +
     0.3 * demand_score +
     0.2 * rent_score -
     0.1 * supply_score) AS investment_score
FROM investment_score;

town,avg_price_sqm,demand,avg_rent,total_supply,price_score,demand_score,rent_score,supply_score,investment_score
bukit batok,5162.990058515181,257320,2618.9594405594407,18410449,0.18425237703543434,0.3228991200851102,0.16913536985759683,0.2659897234092551,0.17779878847030064
bukit timah,6575.0170736445225,24223,2965.3047284740865,1878673,0.5500517989325298,0.0,0.567860560533156,0.0,0.33359283167964315
bukit panjang,5077.840567348026,155059,2695.64531212403,11750069,0.16219356774209792,0.18124141141008024,0.2574189118275622,0.15882684907557584,0.1548509479778181
hougang,5145.005008916741,280872,2619.2855967990504,18823886,0.1795931738648085,0.3555246797287114,0.16951085268749586,0.2726417608719664,0.18513266791483932
bukit merah,6836.173647060537,473595,3004.5856148462462,40574814,0.6177069659772297,0.6224954563588811,0.6130821340759849,0.622605571330965,0.49418729298065667
bishan,6493.773983241169,113308,3035.3342816374643,7271760,0.529004981000397,0.12340556983908861,0.6484811086171633,0.08677263225996101,0.3696426218493219
central area,8311.867864772472,173312,3340.6746877380924,17233445,1.0,0.20652649718515892,1.0,0.24705219555915678,0.637252729599632
ang mo kio,5094.003650693312,746111,2484.372143408123,41553109,0.16638077271794519,1.0,0.014193617820848386,0.6383459501301126,0.3055564376383365
jurong west,4582.415976727783,494261,2683.1698464807782,53508064,0.03384897737610587,0.6511231659204751,0.24305670569702034,0.8306964376893495,0.17441823809705403
pasir ris,4993.009782660671,102315,2909.281416502311,5071676,0.14021732248246385,0.10817744580876812,0.5033645026173125,0.05137415271141598,0.1840756479879369


In [0]:
OPTIMIZE fact_resale;
OPTIMIZE fact_supply;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1776399590770, 1776399591348, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"


In [0]:
CREATE OR REPLACE TABLE workspace.gold.ml_dataset AS
SELECT
    r.resale_price,   -- target

    r.floor_area_sqm,
    r.property_age,

    p.town,
    p.max_floor,
    p.year_completed,

    f.flat_type

FROM workspace.silver.resale_clean r

LEFT JOIN workspace.gold.dim_property p
    ON r.block = p.block
    AND r.street_name = p.street_name

LEFT JOIN workspace.gold.dim_flat_type f
    ON r.flat_type = f.flat_type;

num_affected_rows,num_inserted_rows
